# AValueSARSAJ

> JAX CRLD SARSA agents in value space. Inherits from valuebaseJ.

In [ ]:
#| default_exp Agents/ValueSARSAJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import itertools as it
from functools import partial

import jax
from jax import jit
import jax.numpy as jnp

from fastcore.utils import *

from pyCRLD.Agents.ValueBaseJ import valuebaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class valSARSAJ(valuebaseJ):
    """Class for JAX CRLD SARSA agents in value space."""
    pass

In [ ]:
#| export
@partial(jit, static_argnums=(0, 2))
def RPEisaJ(self: valSARSAJ, Qisa, norm=False):
    """
    Compute temporal-difference reward-prediction error for
    value SARSA dynamics, given joint state-action values `Qisa`.
    """
    Risa = self.valRisaJ(Qisa)
    NextQisa = self.value_NextQisaJ(Qisa)
    n = jnp.newaxis
    E = self.pre[:, n, n] * Risa + self.gamma[:, n, n] * NextQisa - Qisa
    E = E - E.mean(axis=2, keepdims=True) if norm else E
    return E
valSARSAJ.RPEisa = RPEisaJ

In [ ]:
#| export
@partial(jit, static_argnums=0)
def valRisaJ(self: valSARSAJ, Qisa):
    """Average reward Risa given joint state-action values `Qisa`."""
    Xisa = self.strategy_function.action_probabilities(Qisa)
    return self.Risa(Xisa)
valSARSAJ.valRisaJ = valRisaJ

@partial(jit, static_argnums=0)
def valNextQisaJ(self: valSARSAJ, Qisa):
    """Strategy-average next state-action value given joint state-action values `Qisa`."""
    Xisa = self.strategy_function.action_probabilities(Qisa)
    valQisa = self.Qisa(Xisa)

    i = 0; a = 1; s = 2; sprim = 3
    j2k = list(range(4, 4 + self.N - 1))
    b2d = list(range(4 + self.N - 1, 4 + self.N - 1 + self.N))
    e2f = list(range(3 + 2 * self.N, 3 + 2 * self.N + self.N - 1))

    sumsis = [[j2k[l], s, e2f[l]] for l in range(self.N - 1)]
    otherX = list(it.chain(*zip((self.N - 1) * [Xisa], sumsis)))

    NextQisa = jnp.einsum(valQisa, [i, s, a], Xisa, [i, s, a], [i, s])

    args = [self.Omega, [i] + j2k + [a] + b2d + e2f] + otherX + \
           [self.T, [s] + b2d + [sprim], NextQisa, [i, sprim], [i, s, a]]
    return jnp.einsum(*args, optimize=self.opti)
valSARSAJ.value_NextQisaJ = valNextQisaJ

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ
from pyCRLD.Agents.ValueBaseJ import multiagent_epsilongreedy_strategyJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
strat_fn = multiagent_epsilongreedy_strategyJ(epsilon_greedys=0.1, N=2)

agent = valSARSAJ(env, learning_rates=0.1, discount_factors=0.9,
                  strategy_function=strat_fn)

Q = agent.zero_intelligence_values()
assert Q.shape == (2, 1, 2)

rpe = agent.RPEisa(Q)
assert rpe.shape == (2, 1, 2)

In [ ]:
key = jax.random.PRNGKey(2)
Q0 = agent.random_values(key=key)
traj, fpr = agent.trajectory(Q0, Tmax=30)
assert traj.shape == (30, 2, 1, 2)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()